In [ ]:
import subprocess, sys

# Pin numpy<2 FIRST to avoid ABI conflicts with matplotlib and fastai
# opencv>=4.9 requires numpy>=2 which breaks the stack — pin everything together
pkgs = [
    "streamlit",
    "numpy==1.26.4",
    "matplotlib==3.7.5",
    "opencv-python==4.8.1.78",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--force-reinstall",
     "--no-cache-dir", "-q"] + pkgs,
    check=True
)

import cv2, numpy, matplotlib
print(f"✅ numpy       {numpy.__version__}")
print(f"✅ opencv      {cv2.__version__}")
print(f"✅ matplotlib  {matplotlib.__version__}")
print("✅ streamlit   installed")

In [1]:
import importlib

checks = [
    ("numpy",      "1.26"),
    ("matplotlib", "3.7"),
    ("cv2",        "4.8"),
    ("fastai",     None),
    ("streamlit",  None),
    ("PIL",        None),
    ("torch",      None),
]

all_ok = True
for mod, min_ver in checks:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "unknown")
        ok = True
        if min_ver and not ver.startswith(min_ver):
            ok = False
            all_ok = False
        print(f"{'✅' if ok else '⚠️ WRONG VERSION'}  {mod:<20} {ver}")
    except Exception as e:
        all_ok = False
        print(f"❌  {mod:<20} MISSING — {e}")

try:
    import matplotlib._path
    print("✅  matplotlib C extensions  OK (ABI match)")
except Exception as e:
    print(f"❌  matplotlib C extensions  BROKEN — {e}")

try:
    from fastai.learner import load_learner
    from fastai.vision.core import PILImage
    print("✅  fastai imports           OK")
except Exception as e:
    print(f"❌  fastai imports           BROKEN — {e}")

print()
print("ALL GOOD ✅" if all_ok else "ISSUES FOUND ❌ — fix before proceeding")

✅  numpy                1.26.4
✅  matplotlib           3.7.5
✅  cv2                  4.8.1
✅  fastai               2.7.15
✅  streamlit            1.56.0
✅  PIL                  12.2.0
✅  torch                2.2.2+cpu
✅  matplotlib C extensions  OK (ABI match)
✅  fastai imports           OK

ALL GOOD ✅


In [2]:
%%writefile dr_app.py
import os, io, math
os.environ['MPLBACKEND'] = 'Agg'

import cv2
import numpy as np
import streamlit as st
from PIL import Image
from fastai.learner import load_learner
from fastai.vision.core import PILImage

def get_x(r): return r['image']
def get_y(r): return r['label']

MODEL_PATH = '/home/rahulkrithik/Downloads/DR/dr_final/dr_model/model.pkl'

DR_INFO = {
    0: {'name': 'No DR',            'color': '#22c55e', 'base_risk': 0.05},
    1: {'name': 'Mild NPDR',        'color': '#84cc16', 'base_risk': 0.25},
    2: {'name': 'Moderate NPDR',    'color': '#eab308', 'base_risk': 0.50},
    3: {'name': 'Severe NPDR',      'color': '#f97316', 'base_risk': 0.75},
    4: {'name': 'Proliferative DR', 'color': '#ef4444', 'base_risk': 0.95},
}
LEVEL_COLORS = {
    'Low': '#22c55e', 'Moderate': '#eab308',
    'High': '#f97316', 'Critical': '#ef4444',
}

# ── HbA1c classification helper ───────────────────────────────────────────────
def hba1c_category(hba1c):
    if hba1c < 5.7:    return 'Normal',             '#22c55e'
    elif hba1c < 6.5:  return 'Pre-diabetic',        '#eab308'
    elif hba1c < 8.0:  return 'Controlled',          '#84cc16'
    elif hba1c < 10.0: return 'Elevated',            '#f97316'
    else:              return 'Poorly Controlled',   '#ef4444'

@st.cache_resource
def load_model():
    return load_learner(MODEL_PATH)

# ── Risk computation (includes HbA1c factor) ──────────────────────────────────
def compute_risk(stage, confidence, age, hereditary, diabetes_years, hba1c):
    base    = DR_INFO[stage]['base_risk']
    age_f   = min(age / 70.0, 1.0) * 0.15
    hered_f = 0.10 if hereditary else 0.0
    dur_f   = min(math.log1p(diabetes_years) / math.log1p(30), 1.0) * 0.20
    hba1c_f = min((hba1c - 5.0) / 13.0, 1.0) * 0.15
    raw     = base * confidence + age_f + hered_f + dur_f + hba1c_f
    score   = round(min(raw, 1.0) * 100, 1)
    if score < 25:   level = 'Low'
    elif score < 50: level = 'Moderate'
    elif score < 75: level = 'High'
    else:            level = 'Critical'
    return score, level

# ── Lesion segmentation ───────────────────────────────────────────────────────
def calc_lesion(pil_img):
    img_rgb  = np.array(pil_img.convert('RGB'))
    img_bgr  = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    img_gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    img_resized   = cv2.resize(img_rgb,  (512, 512))
    img_gray      = cv2.resize(img_gray, (512, 512))
    green         = img_resized[:, :, 1]
    clahe         = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced      = clahe.apply(green)
    denoised      = cv2.medianBlur(enhanced, 5)
    inv           = 255 - denoised
    _, hem_mask    = cv2.threshold(inv, 225, 255, cv2.THRESH_BINARY)
    _, ret_mask    = cv2.threshold(denoised, 10, 255, cv2.THRESH_BINARY)
    kernel         = np.ones((10, 10), np.uint8)
    inner_ret_mask = cv2.erode(ret_mask, kernel)
    clean_hem_mask = cv2.bitwise_and(hem_mask, inner_ret_mask)
    hem_area  = np.sum(clean_hem_mask > 0)
    ret_area  = np.sum(ret_mask > 0)
    hem_ratio = hem_area / ret_area if ret_area > 0 else 0.0
    thresh_val = np.percentile(denoised, 95)
    ex_mask    = (enhanced > thresh_val).astype(np.uint8) * 255
    _, ex_mask = cv2.threshold(ex_mask, thresh_val, 255, cv2.THRESH_BINARY)
    k2         = np.ones((3, 3), np.uint8)
    ex_mask    = cv2.morphologyEx(ex_mask, cv2.MORPH_OPEN, k2)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(ex_mask)
    clean_ex_mask = np.zeros_like(ex_mask)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] < 1000:
            clean_ex_mask[labels == i] = 255
    clean_ex_mask = cv2.bitwise_and(clean_ex_mask, inner_ret_mask)
    ex_area      = np.sum(clean_ex_mask > 0)
    ex_ratio     = ex_area / ret_area if ret_area > 0 else 0.0
    lesion_score = 0.6 * hem_ratio + 0.4 * ex_ratio
    colored      = img_resized.copy()
    colored[clean_hem_mask > 0] = [255, 0, 0]
    colored[clean_ex_mask  > 0] = [255, 255, 0]
    overlay      = cv2.addWeighted(colored, 0.6, img_resized, 0.4, 0)
    return hem_ratio, ex_ratio, lesion_score, Image.fromarray(overlay)

# ── Clinical advice engine ────────────────────────────────────────────────────
def get_clinical_advice(stage, hem_ratio, ex_ratio, lesion_score,
                        risk_score, age, hereditary, diabetes_years, hba1c):
    advice  = []
    hem_pct = hem_ratio * 100
    ex_pct  = ex_ratio  * 100
    les_pct = lesion_score * 100

    STAGE_ADVICE = {
        0: ('good',    '#22c55e', 'No diabetic retinopathy detected',
            'Retina appears healthy. Maintain annual screening to catch any future changes early.'),
        1: ('info',    '#84cc16', 'Mild changes — monitoring recommended',
            'Early microaneurysms detected. Schedule a follow-up retinal exam within 12 months. No treatment needed at this stage.'),
        2: ('warning', '#eab308', 'Moderate NPDR — closer follow-up required',
            'Retinal changes are progressing. Repeat examination in 3-6 months advised. Strict blood sugar and BP control is essential.'),
        3: ('warning', '#f97316', 'Severe NPDR — urgent ophthalmology referral',
            'Significant retinal ischaemia present. Refer to a retinal specialist within 4 weeks. Anti-VEGF or laser photocoagulation may be needed.'),
        4: ('urgent',  '#ef4444', 'Proliferative DR — immediate specialist intervention',
            'New abnormal blood vessels detected. Sight-threatening condition. Urgent referral within 1 week. PRP laser or vitrectomy likely required.'),
    }
    p, c, t, d = STAGE_ADVICE[stage]
    advice.append({'priority': p, 'color': c, 'title': t, 'detail': d})

    # HbA1c-specific advice
    if hba1c >= 10.0:
        advice.append({'priority': 'urgent', 'color': '#ef4444',
            'title': f'Poorly controlled glycaemia — HbA1c {hba1c:.1f}%',
            'detail': 'HbA1c >= 10% indicates severely uncontrolled diabetes. Immediate endocrinology review required. Aggressive insulin/medication adjustment and dietary intervention are essential to slow retinal progression.'})
    elif hba1c >= 8.0:
        advice.append({'priority': 'warning', 'color': '#f97316',
            'title': f'Elevated HbA1c — {hba1c:.1f}% (target < 7%)',
            'detail': 'Glycaemic control needs improvement. Work with an endocrinologist to adjust therapy. Each 1% reduction in HbA1c is associated with a ~35% reduction in microvascular complication risk.'})
    elif hba1c >= 6.5:
        advice.append({'priority': 'info', 'color': '#84cc16',
            'title': f'HbA1c within controlled range — {hba1c:.1f}%',
            'detail': 'Diabetes is reasonably controlled. Aim to maintain HbA1c < 7% for optimal retinal protection. Continue current management and monitor quarterly.'})
    elif hba1c >= 5.7:
        advice.append({'priority': 'info', 'color': '#eab308',
            'title': f'Pre-diabetic HbA1c — {hba1c:.1f}%',
            'detail': 'HbA1c in the pre-diabetic range. Lifestyle interventions (diet, exercise) are recommended to prevent progression to type 2 diabetes and associated retinal risk.'})
    else:
        advice.append({'priority': 'good', 'color': '#22c55e',
            'title': f'Normal HbA1c — {hba1c:.1f}%',
            'detail': 'Glycated haemoglobin is within the normal range. Maintain healthy lifestyle habits and continue periodic screening.'})

    if hem_pct > 10:
        advice.append({'priority': 'urgent', 'color': '#ef4444',
            'title': 'Extensive haemorrhage — surgical evaluation needed',
            'detail': f'Haemorrhage covers {hem_pct:.2f}% of retinal area. Vitreous surgery (vitrectomy) may be required to prevent permanent vision loss. Do not delay referral.'})
    elif hem_pct > 5:
        advice.append({'priority': 'warning', 'color': '#f97316',
            'title': 'Significant haemorrhage — laser treatment likely',
            'detail': f'Haemorrhage at {hem_pct:.2f}% of retinal area. Laser photocoagulation should be discussed with a retinal specialist. Follow up within 2-4 weeks.'})
    elif hem_pct > 1:
        advice.append({'priority': 'info', 'color': '#eab308',
            'title': 'Mild haemorrhage present',
            'detail': f'Small haemorrhages detected ({hem_pct:.2f}% of retinal area). Monitor every 3-6 months. Optimise glycaemic control.'})

    if ex_pct > 8:
        advice.append({'priority': 'warning', 'color': '#f97316',
            'title': 'Heavy exudate burden — macular oedema risk',
            'detail': f'Exudates cover {ex_pct:.2f}% of retinal area. Central vision at risk. OCT imaging and anti-VEGF injection should be urgently evaluated.'})
    elif ex_pct > 3:
        advice.append({'priority': 'info', 'color': '#eab308',
            'title': 'Moderate exudates — vascular leakage present',
            'detail': f'Lipid exudates at {ex_pct:.2f}% of retinal area. Tighten lipid and BP management. Review at next ophthalmology appointment.'})

    if risk_score >= 75:
        advice.append({'priority': 'urgent', 'color': '#ef4444',
            'title': 'Critical overall risk profile',
            'detail': 'Combined clinical factors place this patient at critical risk. Immediate multidisciplinary review (ophthalmology + endocrinology) strongly recommended.'})
    elif risk_score >= 50:
        advice.append({'priority': 'warning', 'color': '#f97316',
            'title': 'High risk — intensify diabetes management',
            'detail': 'Target HbA1c < 7%, BP < 130/80 mmHg, LDL < 70 mg/dL. Consider ACE inhibitor or ARB therapy if not already prescribed.'})

    if age > 60 and stage >= 2:
        advice.append({'priority': 'info', 'color': '#38bdf8',
            'title': 'Age-related increased vulnerability',
            'detail': 'Patients over 60 with moderate or worse DR face higher rapid progression risk. Increase screening to every 3-4 months.'})

    if diabetes_years >= 15 and stage >= 1:
        advice.append({'priority': 'info', 'color': '#38bdf8',
            'title': f'Long-standing diabetes ({diabetes_years} yrs) — heightened vigilance',
            'detail': 'Cumulative retinal damage risk is high. Ensure renal function and cardiovascular screening are also current.'})

    if hereditary and stage >= 2:
        advice.append({'priority': 'info', 'color': '#818cf8',
            'title': 'Family history amplifies progression risk',
            'detail': 'Genetic predisposition combined with active retinopathy warrants more frequent monitoring. Consider screening first-degree relatives who have diabetes.'})

    if stage <= 1 and les_pct < 1 and risk_score < 30:
        advice.append({'priority': 'good', 'color': '#22c55e',
            'title': 'Low lesion burden — continue current management',
            'detail': 'Lesion score is very low. Keep up diabetes management, healthy diet, regular exercise, and do not miss annual eye exams.'})

    ORDER = {'urgent': 0, 'warning': 1, 'info': 2, 'good': 3}
    advice.sort(key=lambda x: ORDER.get(x['priority'], 9))
    return advice

# ── Page setup ────────────────────────────────────────────────────────────────
st.set_page_config(page_title='DR Vision', page_icon='👁️', layout='wide')

st.markdown('''<style>
[data-testid="stAppViewContainer"]{background:#0a0c10}
[data-testid="stSidebar"]{background:#111318}
[data-testid="stSidebar"] *{color:#e2e8f0}
.stButton>button{background:linear-gradient(135deg,#38bdf8,#818cf8);
  color:#0a0c10;font-weight:700;border:none;
  border-radius:8px;width:100%;padding:12px;font-size:1rem}
h1,h2,h3,p,label,.stMarkdown{color:#e2e8f0!important}
</style>''', unsafe_allow_html=True)

st.markdown('''<div style='text-align:center;padding:30px 0 10px'>
<span style='background:rgba(56,189,248,.1);border:1px solid rgba(56,189,248,.25);
border-radius:999px;padding:5px 18px;font-size:.72rem;letter-spacing:.12em;
color:#38bdf8;text-transform:uppercase'>● AI Retinal Analysis</span>
<h1 style='font-size:2.8rem;font-weight:800;
background:linear-gradient(135deg,#e2e8f0 30%,#38bdf8 100%);
-webkit-background-clip:text;-webkit-text-fill-color:transparent;
margin:12px 0 4px'>DR Vision</h1>
<p style='color:#64748b;margin:0'>Diabetic Retinopathy Detection & Risk Assessment</p>
</div>''', unsafe_allow_html=True)
st.markdown("<hr style='border-color:#1e2230;margin:10px 0 24px'>", unsafe_allow_html=True)

# ── Sidebar ───────────────────────────────────────────────────────────────────
with st.sidebar:
    st.markdown('### Patient Information')
    age            = st.number_input('Age (years)', 1, 120, 50)
    hereditary     = st.radio('Family History of Diabetes', ['No', 'Yes']) == 'Yes'
    diabetes_years = st.number_input('Duration of Diabetes (years)', 0, 80, 5)

    st.markdown("<hr style='border-color:#1e2230;margin:10px 0'>", unsafe_allow_html=True)
    st.markdown('### Glycaemic Control')
    hba1c = st.slider('HbA1c (%)', min_value=5.0, max_value=18.0,
                      value=7.0, step=0.1, format='%.1f')
    hba1c_label, hba1c_color = hba1c_category(hba1c)
    st.markdown(f'''<div style='background:{hba1c_color}18;
border:1px solid {hba1c_color}44;border-radius:10px;
padding:10px 14px;margin-top:6px;display:flex;
align-items:center;justify-content:space-between'>
<span style='color:{hba1c_color};font-size:.82rem;font-weight:700'>{hba1c_label}</span>
<span style='font-family:monospace;color:{hba1c_color};font-size:.82rem'>{hba1c:.1f}%</span>
</div>
<p style='color:#475569;font-size:.7rem;margin:6px 0 0;line-height:1.5'>
Reference ranges: Normal &lt;5.7 · Pre-diabetic 5.7-6.4 · Controlled 6.5-7.9 · Elevated 8.0-9.9 · Poorly Controlled >=10.0
</p>''', unsafe_allow_html=True)

    st.markdown("<hr style='border-color:#1e2230;margin:10px 0'>", unsafe_allow_html=True)
    st.markdown('### Retinal Image')
    uploaded = st.file_uploader('Upload fundus photo', type=['jpg','jpeg','png','tiff'])
    if uploaded:
        st.image(uploaded, caption='Uploaded fundus image', use_container_width=True)
    analyze = st.button('Analyse Retina')

# ── Main panel ────────────────────────────────────────────────────────────────
if analyze:
    if not uploaded:
        st.error('Please upload a retinal image first.')
        st.stop()

    pil_img = Image.open(io.BytesIO(uploaded.read())).convert('RGB')

    with st.spinner('Running DR detection...'):
        learn   = load_model()
        fai_img = PILImage.create(np.array(pil_img))
        pred_class, pred_idx, probs = learn.predict(fai_img)

    with st.spinner('Running lesion segmentation...'):
        hem_ratio, ex_ratio, lesion_score, overlay_pil = calc_lesion(pil_img)

    stage      = int(pred_class)
    confidence = float(probs[pred_idx])
    info       = DR_INFO[stage]
    probs_list = probs.tolist()
    risk_score, risk_level = compute_risk(stage, confidence, age, hereditary, diabetes_years, hba1c)
    risk_color = LEVEL_COLORS[risk_level]

    st.markdown(f'''<div style='background:{info["color"]}18;
border:1px solid {info["color"]}55;border-radius:14px;
padding:28px;text-align:center;margin-bottom:28px'>
<p style='color:{info["color"]};font-size:.7rem;letter-spacing:.14em;
text-transform:uppercase;margin:0 0 6px'>Predicted Stage</p>
<h2 style='color:{info["color"]};font-size:2.2rem;font-weight:800;margin:0 0 6px'>
Stage {stage} — {info["name"]}</h2>
<p style='color:{info["color"]};margin:0;font-family:monospace'>
Confidence: {confidence:.1%}</p>
</div>''', unsafe_allow_html=True)

    col1, col2 = st.columns(2)
    with col1:
        st.markdown('##### Top 3 Predicted Classes')
        top3 = sorted(enumerate(probs_list), key=lambda x: x[1], reverse=True)[:3]
        for rank, (i, p) in enumerate(top3):
            bc   = DR_INFO[i]['color']
            pct  = p * 100
            bold = 'font-weight:700;' if rank == 0 else ''
            crwn = ' 👑' if rank == 0 else ''
            st.markdown(f'''<div style='background:#0d0f14;border:1px solid #1e2230;
border-radius:10px;padding:14px 16px;margin-bottom:10px'>
<div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:8px'>
<span style='color:{bc};font-size:.85rem;{bold}'>{DR_INFO[i]["name"]}{crwn}</span>
<span style='font-family:monospace;color:{bc};font-size:.85rem;{bold}'>{pct:.1f}%</span>
</div>
<div style='height:8px;background:rgba(255,255,255,.06);border-radius:99px;overflow:hidden'>
<div style='width:{pct:.1f}%;height:100%;background:{bc};border-radius:99px'></div>
</div></div>''', unsafe_allow_html=True)

    with col2:
        st.markdown('##### Composite Risk Score')
        hba1c_label_r, hba1c_color_r = hba1c_category(hba1c)
        st.markdown(f'''<div style='background:#0d0f14;border:1px solid #1e2230;
border-radius:12px;padding:26px;text-align:center'>
<div style='font-size:4rem;font-weight:800;color:{risk_color};
font-family:monospace;line-height:1'>{risk_score}
<span style='font-size:1.1rem;opacity:.45'>/100</span></div>
<div style='margin:12px 0'>
<span style='background:{risk_color}22;color:{risk_color};
border:1px solid {risk_color}55;border-radius:999px;
padding:5px 18px;font-size:.85rem;font-weight:700'>{risk_level} Risk</span>
</div>
<hr style='border-color:#1e2230;margin:14px 0'>
<div style='display:flex;flex-wrap:wrap;gap:6px;justify-content:center'>
<span style='background:rgba(255,255,255,.05);border:1px solid #1e2230;
border-radius:999px;padding:3px 12px;font-size:.72rem;font-family:monospace;color:#94a3b8'>
Age: {age} yrs</span>
<span style='background:rgba(255,255,255,.05);border:1px solid #1e2230;
border-radius:999px;padding:3px 12px;font-size:.72rem;font-family:monospace;color:#94a3b8'>
Hereditary: {"Yes" if hereditary else "No"}</span>
<span style='background:rgba(255,255,255,.05);border:1px solid #1e2230;
border-radius:999px;padding:3px 12px;font-size:.72rem;font-family:monospace;color:#94a3b8'>
Diabetes: {diabetes_years} yrs</span>
<span style='background:{hba1c_color_r}18;border:1px solid {hba1c_color_r}44;
border-radius:999px;padding:3px 12px;font-size:.72rem;font-family:monospace;color:{hba1c_color_r}'>
HbA1c: {hba1c:.1f}% · {hba1c_label_r}</span>
</div></div>''', unsafe_allow_html=True)

    st.markdown("<hr style='border-color:#1e2230;margin:24px 0'>", unsafe_allow_html=True)
    st.markdown('##### Lesion Segmentation')
    col3, col4 = st.columns(2)
    with col3:
        st.image(overlay_pil, caption='Haemorrhage (red) · Exudate (yellow)',
                 use_container_width=True)
    with col4:
        hem_pct = hem_ratio * 100
        ex_pct  = ex_ratio  * 100
        les_pct = lesion_score * 100
        lc      = '#f97316' if les_pct > 5 else '#22c55e'
        st.markdown(f'''<div style='background:#0d0f14;border:1px solid #1e2230;
border-radius:12px;padding:22px'>
<div style='margin-bottom:20px'>
<div style='display:flex;justify-content:space-between;margin-bottom:6px'>
<span style='color:#ef4444;font-size:.85rem;font-weight:700'>Haemorrhage</span>
<span style='font-family:monospace;color:#ef4444'>{hem_pct:.3f}%</span></div>
<div style='height:10px;background:rgba(255,255,255,.06);border-radius:99px;overflow:hidden'>
<div style='width:{min(hem_pct*10,100):.1f}%;height:100%;background:#ef4444;border-radius:99px'>
</div></div>
<p style='color:#475569;font-size:.72rem;margin:4px 0 0'>% of retinal area</p></div>
<div style='margin-bottom:20px'>
<div style='display:flex;justify-content:space-between;margin-bottom:6px'>
<span style='color:#eab308;font-size:.85rem;font-weight:700'>Exudates</span>
<span style='font-family:monospace;color:#eab308'>{ex_pct:.3f}%</span></div>
<div style='height:10px;background:rgba(255,255,255,.06);border-radius:99px;overflow:hidden'>
<div style='width:{min(ex_pct*10,100):.1f}%;height:100%;background:#eab308;border-radius:99px'>
</div></div>
<p style='color:#475569;font-size:.72rem;margin:4px 0 0'>% of retinal area</p></div>
<hr style='border-color:#1e2230;margin:16px 0'>
<div style='text-align:center'>
<p style='color:#64748b;font-size:.7rem;letter-spacing:.1em;
text-transform:uppercase;margin:0 0 6px'>Combined Lesion Score</p>
<div style='font-size:2.8rem;font-weight:800;color:{lc};font-family:monospace;line-height:1'>
{les_pct:.3f}<span style='font-size:.9rem;opacity:.5'>%</span></div>
<p style='color:#475569;font-size:.72rem;margin:8px 0 0'>
0.6 x haemorrhage + 0.4 x exudate</p>
</div></div>''', unsafe_allow_html=True)

    st.markdown("<hr style='border-color:#1e2230;margin:28px 0'>", unsafe_allow_html=True)
    st.markdown('##### Clinical Recommendations')
    advice_list = get_clinical_advice(
        stage, hem_ratio, ex_ratio, lesion_score,
        risk_score, age, hereditary, diabetes_years, hba1c
    )
    for adv in advice_list:
        c = adv['color']
        st.markdown(f'''<div style='display:flex;gap:14px;align-items:flex-start;
background:{c}0d;border:1px solid {c}33;border-left:4px solid {c};
border-radius:10px;padding:16px 18px;margin-bottom:12px'>
<div style='flex-shrink:0;width:28px;height:28px;border-radius:50%;
background:{c}22;border:1px solid {c}55;display:flex;align-items:center;
justify-content:center;font-size:.85rem;color:{c};font-weight:700'>{
{"urgent":"✕","warning":"⚠","info":"◎","good":"✓"}.get(adv["priority"],"·")
}</div>
<div>
<p style='color:{c};font-weight:700;font-size:.9rem;margin:0 0 4px'>{adv["title"]}</p>
<p style='color:#94a3b8;font-size:.82rem;margin:0;line-height:1.6'>{adv["detail"]}</p>
</div></div>''', unsafe_allow_html=True)

    st.markdown('''<div style='background:#0d0f14;border:1px solid #1e2230;
border-radius:10px;padding:14px 18px;margin-top:4px'>
<p style='color:#475569;font-size:.75rem;margin:0;line-height:1.6'>
<b style='color:#64748b'>Disclaimer:</b> This tool assists clinical decision-making
and does not replace professional medical advice. All findings must be reviewed
by a qualified ophthalmologist before any treatment decisions are made.</p>
</div>''', unsafe_allow_html=True)

else:
    st.markdown('''<div style='text-align:center;padding:80px 20px;color:#475569'>
<div style='font-size:3.5rem;margin-bottom:16px;opacity:.2'>👁</div>
<p>Upload a retinal image and fill in patient details,<br>
then click <b style='color:#64748b'>Analyse Retina</b> in the sidebar.</p>
</div>''', unsafe_allow_html=True)

Overwriting dr_app.py


In [3]:
import subprocess, sys, time, socket

def port_free(p):
    with socket.socket() as s: return s.connect_ex(('localhost', p)) != 0

PORT = 8501
if not port_free(PORT):
    print(f'Already running → http://localhost:{PORT}')
else:
    subprocess.Popen(
        [sys.executable, '-m', 'streamlit', 'run', 'dr_app.py',
         '--server.port', str(PORT),
         '--server.headless', 'true',
         '--server.enableCORS', 'false'],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    time.sleep(3)
    print(f'✅ Open → http://localhost:{PORT}')

✅ Open → http://localhost:8501
